# Tutor cluster: analysis

Sections follow the dashboard order: app → gateway decisions → placement → hops → workers → overflow → errors → goodput. Numbers only; the answers are in DESIGN.md.

In [ ]:
import json, statistics, sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "metrics"))
import summarise_run, goodput  # the same code the reports use

RUN = "run_j"            # the run to show: run_c ... run_j (gateway runs); baseline_a / baseline_b have no gateway log
RUN_DIR = ROOT / "metrics" / RUN
if not (RUN_DIR / "gateway_requests.jsonl").exists():
    RUN, RUN_DIR = "run_g", ROOT / "metrics" / "run_g"   # fall back to run G if the chosen run is missing


def load(pattern, folder=None):
    return [json.loads(l) for p in sorted((folder or RUN_DIR).glob(pattern)) for l in p.read_text().splitlines() if l.strip()]


gw = load("gateway_requests.jsonl")
traffic = load("traffic_*.jsonl")
S = summarise_run.main(RUN_DIR)
print(f"RUN = {RUN}: {len(traffic)} client requests, {len(gw)} gateway rows")


## 1. App: what the tutor sends

In [ ]:
ok = [r for r in traffic if r["status"] == 200]
for t in sorted({r["tenant"] for r in ok}):
    p = [r["prompt_tokens"] for r in ok if r["tenant"] == t and r.get("prompt_tokens")]
    o = [r["output_tokens"] for r in ok if r["tenant"] == t and r.get("output_tokens")]
    print(f"{t:11}: prompt tokens median {statistics.median(p):.0f}, output tokens median {statistics.median(o):.0f}, n {len(p)}")
print("shared prefix: the 431-token system prompt (DESIGN Part 1); chapters sent (X-Chapter):",
      dict(Counter(r.get("top_chapter") for r in traffic).most_common(10)))


## 2. Gateway decisions: guard, admit, sheds by reason and tenant

In [ ]:
print("decisions:", S["decisions"])
print("guard rejects:", S["guard_rejects"] or "none")
print("sheds by stage:", S["sheds_by_stage"])
print("sheds by tenant/reason:", S["sheds_by_reason_tenant"])
print("admitted by tenant:", S["admitted_by_tenant"])


## 3. Placement: WorkerA / WorkerB

In [ ]:
print("preferred (chapter home):", S["preferred_by_worker"])
print("chosen (placed):        ", S["placed_by_worker"])
print("policy / load bound:", {(r.get("policy"), r.get("load_bound")) for r in gw if r.get("policy")})


## 4. Hops

In [ ]:
print(json.dumps(S["hops"], indent=1))


## 5. Workers: in flight, waiting, KV, preemptions, prefix hit rate

In [ ]:
for w, v in S["workers"].items():
    print(w, {k: v.get(k) for k in ("gateway_in_flight_max", "gateway_in_flight_mean", "gateway_queue_max",
                                     "max_waiting", "max_kv_pct", "preemptions", "prefix_hit_pct", "engine_ttft_mean_ms")})


## 6. Overflow

In [ ]:
print(json.dumps(S.get("overflow", "no overflow in this run"), indent=1))
print("overflow rows by outcome:", dict(Counter(r.get("overflow_outcome") for r in gw if r.get("overflow_outcome"))))


## 7. Errors

In [ ]:
print("client statuses:", dict(Counter(str(r["status"]) for r in traffic)))
print("aborts:", S["aborts"])
print("upstream errors:", dict(Counter(r.get("outcome") for r in gw if str(r.get("outcome", "")).startswith(("upstream", "overflow_unavailable", "leave")))) or "none")


## 8. Goodput across runs

In [ ]:
rows = [r for r in (goodput.run_goodput(ROOT / "metrics" / f, l, u, g) for f, l, u, g in goodput.RUNS) if r]
print(f"{'run':4} {'users':>5} {'offered/min':>11} {'goodput/min':>11} {'share':>6}")
for r in rows:
    print(f"{r['run']:4} {r['users']:5} {r['offered_per_min']:11} {r['goodput_per_min']:11} {r['goodput_share']:6}")


## 9. Estimate vs measured TTFT, with and without the network trip

In [ ]:
print("network offset:", json.loads((ROOT / "metrics" / "network_offset.json").read_text())["offset_s"], "s (run J: its own trip, metrics/run_j/network_offset.json)")
for run in ("run_d", "run_e", "run_f", "run_g", "run_f2", "run_h1", "run_h2", "run_i", "run_j"):
    if (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        e = summarise_run.main(ROOT / "metrics" / run).get("est_vs_measured", {})
        print(f"{run:7} est p50 {e.get('est_p50_s')} s, measured p50 {e.get('measured_p50_s')} s, minus trip {e.get('measured_minus_trip_p50_s')} s; "
              f"within 50 %: {e.get('within_50pct')} raw, {e.get('within_50pct_minus_trip')} minus trip")


## 10. Run J vs run I: the three open decisions

Run J = run I's load with the `over_cap` trigger at 4 s, tenant windows ×2 and no 0.4 s floor. Same code as `metrics/compare_runs.py`. The verdict is in DESIGN.md Part 7, "Run J".

In [ ]:
import compare_runs
gp = json.loads((ROOT / "metrics" / "goodput.json").read_text())
views = {run: compare_runs.run_view(run, gp) for run in ("run_i", "run_j")}
keys = [("goodput / min", "goodput_per_min"), ("goodput interactive", "goodput_interactive"), ("goodput batch", "goodput_batch"),
        ("sheds", "sheds"), ("gateway TTFT ms", "ttft_ms"), ("overflow", "overflow"),
        ("est within 50 % raw", "within_50_raw"), ("est within 50 % minus trip", "within_50_minus_trip"), ("trip s", "trip_s")]
for label, k in keys:
    print(f"{label:28} run I: {views['run_i'][k]}")
    print(f"{'':28} run J: {views['run_j'][k]}")
print("\nfirst 15 s (by arrival):", {r: views[r]["early"] for r in views})
R = [json.loads(l) for l in (ROOT / "metrics" / "run_j" / "gateway_requests.jsonl").read_text().splitlines() if l.strip()]
print("run J timeout_queue by stage:", dict(Counter(r["stage"] for r in R if r.get("reason") == "timeout_queue")))
t0 = min(r["ts"] for r in R)
print("run J batch admitted per minute:", dict(sorted(Counter(int((r["ts"] - t0) // 60) for r in R if r["tenant"] == "batch" and r["decision"] == "ok").items())))
print("\nwhat-if vs run J:")
for w in compare_runs.whatif_check("run_i", views["run_i"], views["run_j"]):
    print(" ", w)


## 11. Runs L and M: guard deaths, client abort, timeout_queue vs late answers

Verdict in DESIGN.md Part 7, "Runs L and M".

In [ ]:
print("L guard:", json.loads((ROOT / "metrics" / "run_l_guard" / "summary.json").read_text()))
print("L abort:", json.loads((ROOT / "metrics" / "run_l_abort" / "summary.json").read_text()))
S = [json.loads(l) for l in (ROOT / "metrics" / "run_l_abort" / "scrape_1s.jsonl").read_text().splitlines() if l.strip()]
for t in sorted({s["t"] for s in S})[3:8]:
    print(f"  t={t:5}", {s["worker"]: (s.get("running"), s.get("kv_pct")) for s in S if s["t"] == t})
M2 = [json.loads(l) for l in (ROOT / "metrics" / "run_m2" / "gateway_requests.jsonl").read_text().splitlines() if l.strip()]
t0 = min(r["ts"] for r in M2)
tq = [r for r in M2 if r.get("reason") == "timeout_queue" and r["stage"] == "admit" and r["ts"] - t0 > 15]
print("M2 admit timeout_queue after 15 s:", len(tq), "est min/median/max:", min(r["est_ttft"] for r in tq), statistics.median(r["est_ttft"] for r in tq), max(r["est_ttft"] for r in tq))
print("  queue depth at the shed:", dict(sorted(Counter(r["queue_depth"] for r in tq).items())), "by worker:", dict(Counter(r["worker"] for r in tq)))
print("late cost:", json.loads((ROOT / "metrics" / "run_m2" / "late_cost.json").read_text())["run_g_late_engine_slot_s"], "engine-slot s for run G's 3 late answers; a shed costs 0")


## Part 5: the nine scrape questions

Numbers only. Pavi's answers are in DESIGN.md Part 5.

### Q5.1. Who sits in your queue vs vLLM's waiting queue?

**What the scrape shows:** In run G the gateway queue held up to 8 requests per worker while vLLM's waiting queue stayed at 0: the waiting moved into the gateway.


In [ ]:
for run in ("run_g", "run_i"):
    s = summarise_run.main(ROOT / "metrics" / run)
    print(run, {w: (v.get("gateway_queue_max"), v.get("max_waiting")) for w, v in s["workers"].items()}, "(gateway queue max, vLLM waiting max)")

### Q5.2. Waiting / running / swapped (or preempted)?

**What the scrape shows:** In run F2 (pressure2 engine) vLLM's waiting queue reached 22 and it preempted 15 requests at KV 99.8 %; on the normal engine preemptions stayed at 0.


In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_f2")
print({w: {k: v.get(k) for k in ("max_running", "max_waiting", "preemptions", "max_kv_pct")} for w, v in s["workers"].items()})

### Q5.3. orch_replica_queue_depth (or your equivalent): which pod, under which mix?

**What the scrape shows:** Our equivalent is `gateway_queue_depth{worker}`: 0 in runs D to F, up to 8 per worker in runs G and I (`mixed`), and 13 in run J.


In [ ]:
for run in ("run_d", "run_g", "run_h1", "run_h2", "run_i"):
    if (ROOT / "metrics" / run / "summary.json").exists() or (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        s = summarise_run.main(ROOT / "metrics" / run)
        print(run, {w: v.get("gateway_queue_max") for w, v in s["workers"].items()}, {m: v.get("sent") for m, v in s["by_mix"].items()})

### Q5.4. If a 32k RAG retrieve and a short agent decode are both ready, who goes first: you or the engine?

**What the scrape shows:** The gateway decides the order between tenants (interactive leaves its queue first): in run G interactive TTFT p50 was 716 ms against 1,369 ms for batch; inside the engine, vLLM's scheduler decides.


In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_g")
print("gateway TTFT by tenant:", s["gateway_ttft_ms"])
src = (ROOT / "control" / "queue.py").read_text()
print([l for l in src.splitlines() if l.startswith("PRIORITY")])

### Q5.5. PagedAttention packs KV; prefix cache reuses KV. Which one saved memory on your shared-prefix mix?

**What the scrape shows:** On one chapter the prefix hit rate was 75.0 % (H1) and 84.9 % (H2), while KV stayed under 12 % on the normal engine; in run F, 30 sequences used 31 % to 36 % of a halved KV.


In [ ]:
for run in ("run_h1", "run_h2", "run_f", "run_f2"):
    if (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        s = summarise_run.main(ROOT / "metrics" / run)
        print(run, {w: (v.get("prefix_hit_pct"), v.get("max_kv_pct")) for w, v in s["workers"].items()}, "(prefix hit %, max KV %)")

### Q5.6. Chunked prefill and continuous batching: engine flags used and why?

**What the scrape shows:** The engine runs with prefix caching and chunked prefill, `--max-num-batched-tokens 4096` and `--max-num-seqs 32` (`ENGINE_ARGS` below).


In [ ]:
src = (ROOT / "cluster" / "modal_vllm.py").read_text()
print(src[src.index("ENGINE_ARGS = ["):src.index("]", src.index("ENGINE_ARGS = [")) + 1])

### Q5.7. KV full after admit: do you shed at the door, or does the engine preempt?

**What the scrape shows:** In run F2 `kv_free` shed 781 requests at the door, and vLLM still preempted 15 inside the engine: both happen.


In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_f2")
print("kv_free sheds:", {k: v for k, v in s["sheds_by_reason_tenant"].items() if "kv_free" in k})
print("preemptions:", {w: v.get("preemptions") for w, v in s["workers"].items()})

### Q5.8. Client gone (aborted): who frees the KV? And how?

**What the scrape shows:** In run L the client closed 10 of 20 streams at 1.0 s; the gateway closed each upstream stream, and vLLM running fell from 19 to 10 and KV from 7.7 % to 4.0 % within about 1.5 s.


In [ ]:
print("aborts in each run:", {run: summarise_run.main(ROOT / "metrics" / run)["aborts"]
       for run in ("run_d", "run_e", "run_f", "run_g", "run_f2") if (ROOT / "metrics" / run).exists()})
print("run L abort:", json.loads((ROOT / "metrics" / "run_l_abort" / "summary.json").read_text()))


### Q5.9. After a worker returns: slam it at 100 %, or ramp while p99 holds?

**What the scrape shows:** A fresh WorkerB answered `/health` with 200, but its first probe took 1,007 ms, then 175 to 187 ms; the gateway sends it traffic only after the warm probe passes.


In [ ]:
print(json.dumps({w: [round(p["engine_ttft_s"] * 1000) for p in v["probes"]]
                  for w, v in json.loads((ROOT / "metrics" / "warm_proof.json").read_text())["workers"].items()}))